# Exploratory Data Analysis: MMRF CoMMpass Clinical Data

**Project:** Biomarker Discovery in Multiple Myeloma Progression  
**Author:** [Your Name]  
**Date:** January 2026  
**Dataset:** MMRF CoMMpass Study

---

## Objectives

In this notebook, we will:

1. **Load** the clinical data files (TSV format)
2. **Explore** the structure and content of each dataset
3. **Identify** missing values and data quality issues
4. **Visualize** key clinical features (age, survival, etc.)
5. **Document** findings for next steps in analysis

## Background

**Multiple Myeloma (MM)** is a cancer of plasma cells, often preceded by a precursor condition called Monoclonal Gammopathy of Undetermined Significance (MGUS). Understanding the clinical factors associated with progression from MGUS to MM is crucial for:

- Early intervention strategies
- Risk stratification of patients
- Personalized treatment approaches

The **CoMMpass study** (Relating Clinical Outcomes in Multiple Myeloma to Personal Assessment of Genetic Profile) is a longitudinal study by the Multiple Myeloma Research Foundation (MMRF) that tracks newly diagnosed MM patients.

---

## 1. Setup and Imports

First, we'll import the necessary Python libraries for data manipulation and visualization.

In [ ]:
# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Statistics
from scipy import stats

# System utilities
import os
import sys
from pathlib import Path

# Add src to path so we can import our custom modules
sys.path.append('../')

# Configure visualization style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 10

# Display settings for pandas
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)

print("✓ All libraries imported successfully!")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

---

## 2. Load Clinical Data Files

We'll use our custom `data_loader` module to load all clinical TSV files. These files contain:

- **clinical.tsv**: Core demographic and diagnostic information
- **exposure.tsv**: Environmental and lifestyle exposures
- **family_history.tsv**: Family medical history
- **follow_up.tsv**: Longitudinal follow-up data
- **pathology_detail.tsv**: Detailed pathology findings

In [ ]:
# Import our custom data loading functions
from src.data_loader import load_clinical_files, merge_clinical_data, validate_data

# Define path to raw data
DATA_DIR = '../data/raw/'

# Load all clinical files
print("Loading clinical data files...")
print("="*60)
data_dict = load_clinical_files(DATA_DIR)

### 2.1 Inspect Individual Files

Let's examine each file separately before merging them.

In [ ]:
# Check which files were successfully loaded
print(f"Successfully loaded {len(data_dict)} files:")
for name in data_dict.keys():
    print(f"  - {name}")

#### Clinical Data (Main File)

In [ ]:
# Examine the clinical dataframe (if loaded)
if 'clinical' in data_dict:
    clinical_df = data_dict['clinical']
    
    print("CLINICAL DATA OVERVIEW")
    print("="*60)
    print(f"Shape: {clinical_df.shape[0]} rows × {clinical_df.shape[1]} columns\n")
    
    # Display basic info
    print("Data Types and Non-Null Counts:")
    clinical_df.info()
else:
    print("⚠ Clinical file not found. Please ensure clinical.tsv is in data/raw/")

In [ ]:
# View first few rows
if 'clinical' in data_dict:
    print("\nFirst 5 rows of clinical data:")
    display(clinical_df.head())

In [ ]:
# Summary statistics for numerical columns
if 'clinical' in data_dict:
    print("\nSummary Statistics (Numerical Columns):")
    display(clinical_df.describe())

---

## 3. Data Quality Assessment

Understanding missing data is crucial for any analysis. Let's identify which columns have missing values and how much data is missing.

In [ ]:
# Function to visualize missing data
def plot_missing_data(df, title="Missing Data Analysis"):
    """
    Create a visualization of missing data patterns.
    
    Parameters:
    -----------
    df : pd.DataFrame
        Input dataframe
    title : str
        Plot title
    """
    # Calculate missing percentages
    missing_pct = (df.isnull().sum() / len(df)) * 100
    missing_pct = missing_pct[missing_pct > 0].sort_values(ascending=False)
    
    if len(missing_pct) == 0:
        print("✓ No missing data found!")
        return
    
    # Create visualization
    fig, ax = plt.subplots(figsize=(12, max(6, len(missing_pct) * 0.3)))
    
    missing_pct.plot(kind='barh', color='coral', ax=ax)
    ax.set_xlabel('Percentage Missing (%)', fontsize=12)
    ax.set_ylabel('Column Name', fontsize=12)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)
    
    # Add percentage labels
    for i, v in enumerate(missing_pct):
        ax.text(v + 1, i, f'{v:.1f}%', va='center', fontsize=9)
    
    plt.tight_layout()
    plt.show()
    
    print(f"\nTotal columns with missing data: {len(missing_pct)} out of {len(df.columns)}")
    print(f"Columns with >50% missing: {(missing_pct > 50).sum()}")

# Apply to clinical data
if 'clinical' in data_dict:
    plot_missing_data(clinical_df, "Missing Data in Clinical File")

---

## 4. Exploratory Visualizations

Now let's create some basic visualizations to understand our patient population.

### 4.1 Age Distribution at Diagnosis

Age is a key factor in cancer prognosis. Let's visualize the age distribution of our patients.

**Note:** Adjust the column name below based on your actual data (e.g., 'age_at_diagnosis', 'age_at_index', 'age', etc.)

In [ ]:
# Find age-related column (different datasets use different names)
if 'clinical' in data_dict:
    age_columns = [col for col in clinical_df.columns if 'age' in col.lower()]
    print(f"Age-related columns found: {age_columns}")
    
    # Try to find the most relevant age column
    age_col = None
    for potential_col in ['age_at_diagnosis', 'age_at_index', 'age']:
        if potential_col in clinical_df.columns:
            age_col = potential_col
            break
    
    if age_col:
        # Remove missing values for visualization
        age_data = clinical_df[age_col].dropna()
        
        # Create visualization
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        
        # Histogram
        axes[0].hist(age_data, bins=30, color='skyblue', edgecolor='black', alpha=0.7)
        axes[0].axvline(age_data.mean(), color='red', linestyle='--', 
                       linewidth=2, label=f'Mean: {age_data.mean():.1f}')
        axes[0].axvline(age_data.median(), color='orange', linestyle='--', 
                       linewidth=2, label=f'Median: {age_data.median():.1f}')
        axes[0].set_xlabel('Age at Diagnosis (years)', fontsize=12)
        axes[0].set_ylabel('Frequency', fontsize=12)
        axes[0].set_title('Age Distribution', fontsize=14, fontweight='bold')
        axes[0].legend()
        axes[0].grid(alpha=0.3)
        
        # Box plot
        axes[1].boxplot(age_data, vert=True)
        axes[1].set_ylabel('Age at Diagnosis (years)', fontsize=12)
        axes[1].set_title('Age Distribution (Box Plot)', fontsize=14, fontweight='bold')
        axes[1].grid(alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        # Print statistics
        print(f"\nAge Statistics:")
        print(f"  Mean: {age_data.mean():.2f} years")
        print(f"  Median: {age_data.median():.2f} years")
        print(f"  Std Dev: {age_data.std():.2f} years")
        print(f"  Range: {age_data.min():.0f} - {age_data.max():.0f} years")
        print(f"  Patients with age data: {len(age_data)}")
    else:
        print("⚠ No standard age column found. Please check column names in your dataset.")

In [ ]:
# Find gender/sex column
if 'clinical' in data_dict:
    gender_columns = [col for col in clinical_df.columns if any(word in col.lower() for word in ['gender', 'sex'])]
    print(f"Gender-related columns found: {gender_columns}")
    
    # Try common column names
    gender_col = None
    for potential_col in ['gender', 'sex', 'Gender', 'Sex']:
        if potential_col in clinical_df.columns:
            gender_col = potential_col
            break
    
    if gender_col:
        gender_counts = clinical_df[gender_col].value_counts()
        
        # Create pie chart
        fig, ax = plt.subplots(figsize=(8, 8))
        colors = ['lightblue', 'lightcoral', 'lightgray']
        
        gender_counts.plot(kind='pie', autopct='%1.1f%%', colors=colors[:len(gender_counts)], 
                          startangle=90, ax=ax, textprops={'fontsize': 12})
        ax.set_ylabel('')
        ax.set_title('Gender Distribution', fontsize=14, fontweight='bold', pad=20)
        
        plt.tight_layout()
        plt.show()
        
        print(f"\nGender Distribution:")
        for gender, count in gender_counts.items():
            print(f"  {gender}: {count} ({count/len(clinical_df)*100:.1f}%)")
    else:
        print("⚠ No standard gender column found.")

### 4.3 Survival Status

One of the most important outcomes in cancer research is patient survival.

In [ ]:
# Look for survival-related columns
if 'clinical' in data_dict:
    survival_columns = [col for col in clinical_df.columns if any(word in col.lower() 
                        for word in ['vital', 'status', 'survival', 'death', 'alive'])]
    print(f"Survival-related columns found: {survival_columns[:10]}")  # Show first 10
    
    # Common column names for vital status
    status_col = None
    for potential_col in ['vital_status', 'vital_state', 'status', 'alive_dead']:
        if potential_col in clinical_df.columns:
            status_col = potential_col
            break
    
    if status_col:
        status_counts = clinical_df[status_col].value_counts()
        
        # Create bar chart
        fig, ax = plt.subplots(figsize=(10, 6))
        
        status_counts.plot(kind='bar', color=['green', 'red', 'gray'][:len(status_counts)], 
                          edgecolor='black', alpha=0.7, ax=ax)
        ax.set_xlabel('Vital Status', fontsize=12)
        ax.set_ylabel('Number of Patients', fontsize=12)
        ax.set_title('Patient Vital Status Distribution', fontsize=14, fontweight='bold')
        ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
        ax.grid(axis='y', alpha=0.3)
        
        # Add count labels on bars
        for i, v in enumerate(status_counts):
            ax.text(i, v + max(status_counts)*0.02, str(v), 
                   ha='center', va='bottom', fontweight='bold')
        
        plt.tight_layout()
        plt.show()
        
        print(f"\nVital Status Distribution:")
        for status, count in status_counts.items():
            print(f"  {status}: {count} ({count/len(clinical_df)*100:.1f}%)")
    else:
        print("⚠ No standard vital status column found.")

---

## 5. Merge All Clinical Files

Now let's merge all the clinical files into a single comprehensive dataset.

In [ ]:
# Merge all files if we have data
if data_dict:
    merged_df = merge_clinical_data(data_dict, merge_key='case_id')
else:
    print("⚠ No data to merge. Please add data files to data/raw/")

In [ ]:
# Validate the merged data
if 'merged_df' in locals():
    is_valid = validate_data(merged_df, required_columns=['case_id'])

---

## 6. Save Processed Data

Let's save our merged dataset for use in future analyses.

In [ ]:
# Save merged data to processed directory
if 'merged_df' in locals():
    output_path = '../data/processed/clinical_merged.csv'
    merged_df.to_csv(output_path, index=False)
    print(f"✓ Merged data saved to: {output_path}")
    print(f"  Shape: {merged_df.shape}")
    print(f"  Size: {os.path.getsize(output_path) / (1024*1024):.2f} MB")
else:
    print("⚠ No merged data to save.")

---

## 7. Key Findings and Next Steps

### Summary of Findings

**Data successfully loaded** (update based on your actual results):
- ✓ Clinical data: [X rows, Y columns]
- ✓ Follow-up data: [X rows, Y columns]
- ✓ Pathology data: [X rows, Y columns]

**Key Observations** (fill in after running):
1. Patient cohort size: [N] patients
2. Age distribution: Mean ~[X] years
3. Gender distribution: [%] male, [%] female
4. Survival status: [%] alive, [%] deceased
5. Missing data: Most complete variables are [list], most incomplete are [list]

### Next Steps

1. **Data Cleaning**: Handle missing values appropriately (imputation vs. exclusion)
2. **Feature Engineering**: Create derived variables (e.g., time to event, risk scores)
3. **Survival Analysis**: Kaplan-Meier curves, Cox proportional hazards models
4. **Biomarker Discovery**: Statistical tests and machine learning for feature selection
5. **Genomic Integration**: Merge with RNA-seq/mutation data if available

### Questions to Address

- What clinical factors are associated with better/worse outcomes?
- Can we identify distinct patient subgroups?
- Which features are most predictive of progression from MGUS to MM?
- Are there any unexpected patterns in the data?

---

**Notebook completed!** 🎉

Ready to move on to more advanced analyses in subsequent notebooks.